In [ ]:
# Cell 148 (05_convergence_checks.ipynb, cell 1): Phase 5b Part A. Converged LR pre-check. FRESH KERNEL, only this cell. Writes 3 sel_ files (exclusive create).
def _c148():
    import os, sys, json, time, pickle, hashlib, warnings
    import numpy as np, pandas as pd
    from sklearn.linear_model import LogisticRegression
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.metrics import roc_auc_score, average_precision_score
    from scipy.stats import ks_2samp
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    O = {"t": "selection/tables/sel_precheck_converged_table.csv", "b": "selection/tables/sel_precheck_converged_bootstrap.csv",
         "n": "selection/tables/sel_precheck_converged_notes.json"}
    for rel in O.values(): ss.check_path(os.path.join(R, rel))
    PROC = R + "/Data/processed/"; L = lambda p: pickle.load(open(p, "rb"))
    nb = json.load(open(R + "/selection/notebooks/00_precheck.ipynb")); code = "".join(nb["cells"][2]["source"])
    g = {"os": os, "np": np, "pd": pd, "L": L, "PROC": PROC, "ROOT": R}
    sp = L(PROC + "split_indices.pkl")
    g["train_idx"], g["test_idx"] = np.asarray(sp["train_idx"]), np.asarray(sp["test_idx"])
    y_tr, y_te = np.asarray(sp["y_train"]).astype(int), np.asarray(sp["y_test"]).astype(int)
    g["y_train"], g["y_test"] = y_tr, y_te
    exec(compile(code[:code.index("# ---- validation")], "build_design_src", "exec"), g)
    bd = g["build_design"]
    SPEC = {"A": ['Age', 'Debt Ratio', 'Monthly Income', 'Open Credit Lines', 'Real Estate Loans', 'Number of Dependents'],
            "B": ['Age', 'Debt Ratio', 'Open Credit Lines', 'Real Estate Loans']}
    DS = {k: bd(v) for k, v in SPEC.items()}
    old = L(R + "/selection/predictions/sel_precheck_predictions.pkl")
    P, info = {}, []
    for k in "AB":
        Xtr, Xte = DS[k]["Xtr"].values, DS[k]["Xte"].values
        for C in (100, 1):
            with warnings.catch_warnings(record=True) as w:
                warnings.simplefilter("always")
                m = LogisticRegression(C=C, penalty="l2", solver="lbfgs", class_weight="balanced", max_iter=5000).fit(Xtr, y_tr)
            conv = [x for x in w if issubclass(x.category, ConvergenceWarning)]
            ni = int(np.max(m.n_iter_))
            print("Spec %s lbfgs C=%-3d n_iter_ %d (max_iter 5000) | ConvergenceWarnings %d" % (k, C, ni, len(conv)))
            if C == 100: assert ni < 5000 and not conv, "locked-C lbfgs did not converge: STOP"
            P[(k, C)] = m.predict_proba(Xte)[:, 1]; info.append(dict(spec=k, C=C, n_iter=ni, converged=(ni < 5000 and not conv)))
    n = len(y_te); B = 2000; rng = np.random.RandomState(42); boot = {k: np.empty(B) for k in P}
    for i in range(B):
        ix = rng.randint(0, n, n); yy = y_te[ix]
        for k in P: boot[k][i] = roc_auc_score(yy, P[k][ix])
    rows = []
    for (k, C), p in P.items():
        a = roc_auc_score(y_te, p); lo, hi = np.percentile(boot[(k, C)], [2.5, 97.5])
        rows.append(dict(spec=k, vars=len(SPEC[k]), C=C, solver="lbfgs", n_iter=[i for i in info if i["spec"] == k and i["C"] == C][0]["n_iter"],
                         primary=(C == 100), auc=a, auc_ci_lo=lo, auc_ci_hi=hi, gini=2 * a - 1, ks=float(ks_2samp(p[y_te == 1], p[y_te == 0]).statistic),
                         ap=float(average_precision_score(y_te, p)), phase1_saga_auc=(roc_auc_score(y_te, old[k]) if C == 100 else np.nan)))
    tbl = pd.DataFrame(rows)
    brow = []
    for C in (100, 1):
        d = boot[("A", C)] - boot[("B", C)]; obs = roc_auc_score(y_te, P[("A", C)]) - roc_auc_score(y_te, P[("B", C)]); lo, hi = np.percentile(d, [2.5, 97.5])
        brow.append(dict(comparison="A - B", C=C, primary=(C == 100), diff=obs, ci_lo=lo, ci_hi=hi, p_value=max(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())), 1 / 2001), n_resamples=B))
    bt = pd.DataFrame(brow)
    pd.set_option("display.width", 220, "display.max_columns", 30)
    print("\nTEST (n=%d), converged lbfgs LR:" % n); print(tbl.to_string(index=False, float_format=lambda x: "%.4f" % x))
    print("\nPaired bootstrap A - B (2,000, seed 42, p floor 0.0005):"); print(bt.to_string(index=False, float_format=lambda x: "%.4f" % x))
    notes = dict(purpose="Phase 1 pre-check redone with a converged LR (lbfgs, max_iter 5000, locked C=100 l2 balanced; C=1 for reference)",
                 superseded_phase1_files=["selection/tables/sel_precheck_table.csv", "selection/tables/sel_precheck_bootstrap.csv",
                                          "selection/tables/sel_precheck_notes.json", "selection/predictions/sel_precheck_predictions.pkl"],
                 reason="Phase 1 used saga at C=100, which hit max_iter=1000 (not converged)",
                 convergence=info, pgmpy_note="n/a", seed=42)
    def wr(rel, data):
        p = ss.check_path(os.path.join(R, rel))
        with open(p, "xb") as f: f.write(data)
        assert hashlib.sha256(open(p, "rb").read()).hexdigest() == hashlib.sha256(data).hexdigest()
        return hashlib.sha256(data).hexdigest()[:12]
    print("\nsaved:", wr(O["t"], tbl.to_csv(index=False).encode()), wr(O["b"], bt.to_csv(index=False).encode()), wr(O["n"], json.dumps(notes, indent=1).encode()))
_c148()

In [ ]:
# Cell 149 (05_convergence_checks.ipynb): Part B inspection. READ-ONLY, writes NOTHING. No restart.
def _c149():
    import os, re, json, pickle, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    R = os.environ["CREDIT_BN_ROOT"]
    FV = R + "/full_variable"
    for d in ("models", "predictions", "tables"):
        print(d + ":", sorted(os.listdir(FV + "/" + d)))
    lr = pickle.load(open(FV + "/models/lr_final.pkl", "rb"))
    print("\nlr_final:", {k: lr.get_params()[k] for k in ("solver", "C", "penalty", "max_iter", "class_weight", "random_state", "tol")},
          "| n_iter_", lr.n_iter_, "| reached max_iter:", int(np.max(lr.n_iter_)) >= lr.max_iter, "| n_features", lr.coef_.shape[1])
    for f in sorted(os.listdir(FV + "/predictions")):
        try:
            o = pickle.load(open(FV + "/predictions/" + f, "rb"))
            print("\n", f, type(o).__name__, list(o.keys()) if isinstance(o, dict) else getattr(o, "shape", ""))
            if isinstance(o, dict):
                for k, v in o.items():
                    print("    ", k, type(v).__name__, getattr(v, "shape", len(v) if hasattr(v, "__len__") else v))
        except Exception as e:
            print(f, "unreadable:", type(e).__name__)
    nb = json.load(open(FV + "/notebooks/Expirements_FROZEN.ipynb"))
    for ci, c in enumerate(nb["cells"]):
        s = "".join(c["source"]) if isinstance(c["source"], list) else c["source"]
        if c.get("cell_type") == "code" and ci in (72, 73):
            print("\n--- frozen cell %d, lines 40-100 ---" % ci); print("\n".join(s.splitlines()[40:100]))
    for f in ("X_train_df.pkl", "X_test_df.pkl", "split_indices.pkl"):
        print(f, os.path.exists(R + "/Data/processed/" + f))
_c149()

In [ ]:
# Cell 150 (05_convergence_checks.ipynb): Phase 5b Part B. Convergence check of the LOCKED full-variable LR. Read-only on full_variable/. No restart.
def _c150():
    import os, sys, re, json, pickle, hashlib, importlib.util, warnings
    import numpy as np, pandas as pd
    from sklearn.linear_model import LogisticRegression
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.metrics import roc_auc_score, average_precision_score
    from scipy.stats import ks_2samp
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    O_C, O_J = "selection/tables/sel_lockedLR_convergence_check.csv", "selection/tables/sel_lockedLR_convergence_check.json"
    for rel in (O_C, O_J): ss.check_path(os.path.join(R, rel))
    spec = importlib.util.spec_from_file_location("sens", R + "/shared/src/sensitivity.py")
    sens = importlib.util.module_from_spec(spec); spec.loader.exec_module(sens)
    L = lambda p: pickle.load(open(p, "rb")); FV = R + "/full_variable"; PROC = R + "/Data/processed/"

    # 1. locked objects
    lr = L(FV + "/models/lr_final.pkl")
    prm = {k: lr.get_params()[k] for k in ("solver", "C", "penalty", "max_iter", "class_weight", "random_state", "tol")}
    n_it = int(np.max(lr.n_iter_)); reached = n_it >= lr.max_iter
    print("locked lr_final:", prm, "| n_iter_", n_it, "| reached max_iter:", reached)
    nb = json.load(open(FV + "/notebooks/Expirements_FROZEN.ipynb")); abl_src = {}
    for ci in (67, 72):
        s = "".join(nb["cells"][ci]["source"]) if isinstance(nb["cells"][ci]["source"], list) else nb["cells"][ci]["source"]
        m = re.search(r"make_lr = lambda: (LogisticRegression\([^)]*\))", s, re.S)
        abl_src[ci] = " ".join(m.group(1).split()) if m else "not found"
    print("ablation LRs (4-blanket, no Income/Dependents): fitted objects NOT saved (only predictions). Frozen cells 67/72 construct:", abl_src[67])
    print("  -> solver and max_iter read from the frozen notebook JSON; their n_iter_ is unknown.")

    # 2. refit lbfgs on the locked 35-column design
    sp = L(PROC + "split_indices.pkl"); y_tr, y_te = np.asarray(sp["y_train"]).astype(int), np.asarray(sp["y_test"]).astype(int)
    Xtr, Xte = L(PROC + "X_train_df.pkl"), L(PROC + "X_test_df.pkl")
    assert Xtr.shape == (119999, 35) and Xte.shape == (30000, 35) and lr.coef_.shape[1] == 35
    abl = L(FV + "/predictions/ablation_predictions.pkl")
    p_obj = lr.predict_proba(Xte.values)[:, 1]; p_file = np.asarray(abl["LR_all"], float)
    print("locked LR: object vs saved predictions max |diff| = %.3e" % np.max(np.abs(p_obj - p_file)))
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        m = LogisticRegression(C=100, penalty="l2", solver="lbfgs", class_weight="balanced", max_iter=5000).fit(Xtr.values, y_tr)
    conv_w = [x for x in w if issubclass(x.category, ConvergenceWarning)]
    ni = int(np.max(m.n_iter_)); assert ni < 5000 and not conv_w, "lbfgs did not converge: STOP"
    print("converged lbfgs refit: n_iter_ %d (max_iter 5000), ConvergenceWarnings %d" % (ni, len(conv_w)))
    p_new = m.predict_proba(Xte.values)[:, 1]
    mx = float(np.max(np.abs(p_new - p_obj))); print("max |converged - locked| probability = %.3e | max |coef diff| = %.3e" % (mx, np.max(np.abs(m.coef_ - lr.coef_))))

    def met(p):
        a = roc_auc_score(y_te, p); return dict(auc=a, gini=2 * a - 1, ks=float(ks_2samp(p[y_te == 1], p[y_te == 0]).statistic), ap=float(average_precision_score(y_te, p)))
    TAU = 0.69
    def dec(p):
        pr = p >= TAU; TP, FP, FN, TN = int((pr & (y_te == 1)).sum()), int((pr & (y_te == 0)).sum()), int((~pr & (y_te == 1)).sum()), int((~pr & (y_te == 0)).sum())
        return dict(tau=TAU, TP=TP, FP=FP, FN=FN, TN=TN, cost=5 * FN + FP, flagged=TP + FP)
    ml, mc = met(p_file), met(p_new); dl, dc = dec(p_file), dec(p_new)
    print("\nlocked expected AUC/Gini/KS/AP 0.8618/0.7236/0.5703/0.3771; locked saga reproduced: %.4f/%.4f/%.4f/%.4f" % (ml["auc"], ml["gini"], ml["ks"], ml["ap"]))
    print("converged lbfgs:                                                      %.4f/%.4f/%.4f/%.4f" % (mc["auc"], mc["gini"], mc["ks"], mc["ap"]))
    print("decisions at the locked tau 0.69 (no sweep): locked %s | converged %s" % (dl, dc))
    assert (dl["TP"], dl["FP"], dl["FN"], dl["TN"], dl["cost"]) == (1139, 2143, 866, 25852, 6473), "locked decision counts not reproduced: STOP"

    # 3. paired bootstraps, locked alignment (30,000 rows for LR/RF, 29,998 BN-valid rows for BN)
    p_bn = np.asarray(abl["BN"], float); v = np.isfinite(p_bn); print("BN-valid rows:", int(v.sum()))
    assert int(v.sum()) == 29998
    p_rf = np.asarray(abl["RF_all"], float)
    comps = [("converged - locked LR (30,000 rows)", y_te, p_new, p_file), ("converged LR - BN (29,998 rows)", y_te[v], p_new[v], p_bn[v]),
             ("converged LR - RF (30,000 rows)", y_te, p_new, p_rf), ("LOCKED LR - BN (29,998 rows)", y_te[v], p_file[v], p_bn[v]),
             ("LOCKED LR - RF (30,000 rows)", y_te, p_file, p_rf)]
    rows = []
    for lab, y, a, b in comps:
        obs, lo, hi, p, nbt = sens.paired_bootstrap(y, a, b, n_boot=2000, seed=42)
        rows.append(dict(comparison=lab, diff=obs, ci_lo=lo, ci_hi=hi, p=p, significant=bool(lo > 0 or hi < 0), n_boot=nbt))
    bt = pd.DataFrame(rows); pd.set_option("display.width", 220)
    print("\nPaired bootstrap (2,000, seed 42; p floor 1/n_boot):"); print(bt.to_string(index=False, float_format=lambda x: "%.4f" % x))

    # 4. STOP rule
    d_auc = mc["auc"] - 0.8618
    sig_change = {"LR-BN": bool(bt.significant[1]) != bool(bt.significant[3]) or np.sign(bt["diff"][1]) != np.sign(bt["diff"][3]),
                  "LR-RF": bool(bt.significant[2]) != bool(bt.significant[4]) or np.sign(bt["diff"][2]) != np.sign(bt["diff"][4])}
    stop = abs(d_auc) > 0.002 or any(sig_change.values())
    print("\nconverged AUC - 0.8618 = %+.5f (threshold 0.002) | significance/sign change vs locked: %s" % (d_auc, sig_change))
    print("STOP RULE:", "TRIGGERED, files NOT saved, report to planning instance" if stop else "not triggered")
    if stop: return None

    tbl = pd.DataFrame([dict(model="locked LR (saga, C=100, l2, max_iter 1000, n_iter_ %d)" % n_it, **ml, **dl), dict(model="converged LR (lbfgs, C=100, l2, max_iter 5000, n_iter_ %d)" % ni, **mc, **dc)])
    info = dict(locked_lr=dict(params=prm, n_iter=n_it, reached_max_iter=bool(reached)), ablation_lr_objects_saved=False, ablation_construction_from_frozen_cells=abl_src,
                converged=dict(solver="lbfgs", C=100, penalty="l2", max_iter=5000, n_iter=ni, convergence_warnings=0),
                max_abs_prob_diff_converged_vs_locked=mx, auc_diff_vs_0_8618=d_auc, significance_change=sig_change,
                bootstrap=rows, tau_locked=TAU, note="no file in full_variable/ changed; no sweep on test")
    def wr(rel, data):
        pth = ss.check_path(os.path.join(R, rel))
        with open(pth, "xb") as f: f.write(data)
        assert hashlib.sha256(open(pth, "rb").read()).hexdigest() == hashlib.sha256(data).hexdigest()
        return hashlib.sha256(data).hexdigest()[:12]
    print("saved:", wr(O_C, tbl.to_csv(index=False).encode()), wr(O_J, json.dumps(info, indent=1, default=float).encode()))
_c150()

In [ ]:
# Cell 151 (05_convergence_checks.ipynb): Phase 5b manifest/git DRY RUN. Writes NOTHING. No restart. Save the notebook first (Cmd+S).
def _c151():
    import os, csv, time, hashlib, subprocess
    R = os.environ["CREDIT_BN_ROOT"]
    TAG12 = "5626192403ef"
    def git(*a):
        return subprocess.run(["git", "-C", R, *a], capture_output=True, text=True).stdout.strip()
    def sha(p):
        h = hashlib.sha256()
        with open(p, 'rb') as f:
            for b in iter(lambda: f.read(1 << 20), b''):
                h.update(b)
        return h.hexdigest()
    mp = os.path.join(R, "MANIFEST.csv")
    raw = open(mp, 'rb').read()
    existing = {r['path'] for r in csv.DictReader(open(mp, newline=''))}
    print("DRY_RUN = True")
    print("manifest rows now:", len(existing), "(expected 170) | ends with newline:", raw.endswith(b'\n'))
    tag = git("rev-list", "-n", "1", "full-variable-final")
    print("tag points to:", tag[:12], "| OK:", tag.startswith(TAG12))
    print("git status before:", git("status", "--porcelain").splitlines())
    rows = []
    for top in ("selection", "shared"):
        for dp, dn, fn in os.walk(os.path.join(R, top)):
            dn[:] = [d for d in dn if d not in (".ipynb_checkpoints", "__pycache__")]
            for f in sorted(fn):
                if f.endswith(".pyc") or f == ".DS_Store":
                    continue
                p = os.path.join(dp, f); rel = os.path.relpath(p, R)
                if rel not in existing:
                    rows.append((rel, os.path.getsize(p), sha(p)[:12], time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(p)))))
    print("\nrows to append (%d):" % len(rows))
    for r in sorted(rows): print("  ", *r)
    print("expected manifest rows after:", len(existing) + len(rows))
    print("\nwould stage ONLY: MANIFEST.csv, selection/notebooks/05_convergence_checks.ipynb (not 03_em_score.ipynb, Reminder.md, Writing_Edits.md)")
    print("DRY RUN ONLY: nothing written.")
_c151()

In [ ]:
# Cell 152 (05_convergence_checks.ipynb): REAL RUN (approved). Manifest append (6 Phase 5b files) + NEW git commit. Tag never touched.
def _c152():
    import os, csv, time, hashlib, subprocess
    R = os.environ["CREDIT_BN_ROOT"]
    TAG12 = "5626192403ef"
    NOTE = "added after Phase 5b, 2026-10-15"
    NB = "selection/notebooks/05_convergence_checks.ipynb"
    F = [(NB, None),
         ("selection/tables/sel_lockedLR_convergence_check.csv", "efcb9a233717"),
         ("selection/tables/sel_lockedLR_convergence_check.json", "f95c236058bd"),
         ("selection/tables/sel_precheck_converged_bootstrap.csv", "b73471d0d26e"),
         ("selection/tables/sel_precheck_converged_notes.json", "2fda7ca17872"),
         ("selection/tables/sel_precheck_converged_table.csv", "b24d49cd1b8c")]
    def git(*a, check=True):
        r = subprocess.run(["git", "-C", R, *a], capture_output=True, text=True)
        if check and r.returncode != 0:
            raise RuntimeError(f"git {' '.join(a)} failed: {r.stderr.strip()[:300]}")
        return r.stdout.strip()
    def sha(p):
        h = hashlib.sha256()
        with open(p, 'rb') as f:
            for b in iter(lambda: f.read(1 << 20), b''):
                h.update(b)
        return h.hexdigest()
    mp = os.path.join(R, "MANIFEST.csv")
    raw0 = open(mp, 'rb').read()
    existing = {r['path'] for r in csv.DictReader(open(mp, newline=''))}
    assert raw0.endswith(b'\n') and len(existing) == 170, "unexpected manifest state: STOP"
    assert git("rev-list", "-n", "1", "full-variable-final").startswith(TAG12), "tag moved: STOP"
    rows = []
    for rel, h12 in F:
        p = os.path.join(R, rel)
        assert rel not in existing and os.path.exists(p), f"unexpected state for {rel}: STOP"
        h = sha(p)
        if h12 is not None:
            assert h.startswith(h12), f"hash changed since dry run for {rel}: STOP"
        else:
            print("notebook hash recorded now: %s (dry run showed 2eeb4fee0a88)" % h[:12])
        rows.append([rel, os.path.getsize(p), h, "selection", time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(p))), NOTE])
    with open(mp, 'a', newline='') as fh:
        csv.writer(fh).writerows(rows)
    n = len(list(csv.DictReader(open(mp, newline=''))))
    assert n == 176 and open(mp, 'rb').read().startswith(raw0), "manifest check failed: STOP (not committed)"
    git("add", "MANIFEST.csv", NB)
    ident = []
    if not git("config", "user.name", check=False) or not git("config", "user.email", check=False):
        ident = ["-c", "user.name=Ahmed Nanabhay", "-c", "user.email=noreply@local.invalid"]
    git(*ident, "commit", "-q", "-m", "Phase 5b: converged-LR pre-check and locked-LR convergence check added to manifest")
    print("manifest rows now:", n, "(expected 176)")
    print("new commit:", git("log", "--oneline", "-1"))
    print("tag unchanged:", git("rev-list", "-n", "1", "full-variable-final").startswith(TAG12))
    print("git status:", git("status", "--porcelain").splitlines())
_c152()